In [13]:
import pandas as pd

matches = pd.read_csv("Data/Matches.csv", parse_dates=["MatchDate"])
elo = pd.read_csv("Data/EloRatings.csv", parse_dates=["date"])

matches.head()


C:\Users\Admin\AppData\Local\Temp\ipykernel_17828\2730841688.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  matches = pd.read_csv("Data/Matches.csv", parse_dates=["MatchDate"])


,Division,MatchDate,MatchTime,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,...,MaxUnder25,HandiSize,HandiHome,HandiAway,C_LTH,C_LTA,C_VHD,C_VAD,C_HTB,C_PHB
0,F1,2000-07-28,NaN,Marseille,Troyes,1686.34,1586.57,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F1,2000-07-28,NaN,Paris SG,Strasbourg,1714.89,1642.51,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F2,2000-07-28,NaN,Wasquehal,Nancy,1465.08,1633.80,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,F1,2000-07-29,NaN,Auxerre,Sedan,1635.58,1624.22,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,F1,2000-07-29,NaN,Bordeaux,Metz,1734.34,1673.11,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
elo.head()

,date,club,country,elo
0,2000-07-01,Aachen,GER,1453.60
1,2000-07-01,Aalborg,DEN,1482.61
2,2000-07-01,Aalst,BEL,1337.53
3,2000-07-01,Aarhus,DEN,1381.46
4,2000-07-01,Aberdeen,SCO,1360.43


In [15]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230557 entries, 0 to 230556
Data columns (total 48 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Division     230557 non-null  object        
 1   MatchDate    230557 non-null  datetime64[ns]
 2   MatchTime    99072 non-null   object        
 3   HomeTeam     230557 non-null  object        
 4   AwayTeam     230557 non-null  object        
 5   HomeElo      141597 non-null  float64       
 6   AwayElo      141528 non-null  float64       
 7   Form3Home    229057 non-null  float64       
 8   Form5Home    229057 non-null  float64       
 9   Form3Away    229057 non-null  float64       
 10  Form5Away    229057 non-null  float64       
 11  FTHome       230554 non-null  float64       
 12  FTAway       230554 non-null  float64       
 13  FTResult     230554 non-null  object        
 14  HTHome       175977 non-null  float64       
 15  HTAway       175977 non-null  floa

In [16]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230557 entries, 0 to 230556
Data columns (total 48 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Division     230557 non-null  object        
 1   MatchDate    230557 non-null  datetime64[ns]
 2   MatchTime    99072 non-null   object        
 3   HomeTeam     230557 non-null  object        
 4   AwayTeam     230557 non-null  object        
 5   HomeElo      141597 non-null  float64       
 6   AwayElo      141528 non-null  float64       
 7   Form3Home    229057 non-null  float64       
 8   Form5Home    229057 non-null  float64       
 9   Form3Away    229057 non-null  float64       
 10  Form5Away    229057 non-null  float64       
 11  FTHome       230554 non-null  float64       
 12  FTAway       230554 non-null  float64       
 13  FTResult     230554 non-null  object        
 14  HTHome       175977 non-null  float64       
 15  HTAway       175977 non-null  floa

In [30]:
matches.isna().sum().sort_values(ascending=False).head(15)


MatchTime      131485
C_LTA          117955
C_VHD          117955
C_VAD          117955
C_LTH          117955
C_HTB          117955
C_PHB          117955
HomeTarget     116628
AwayTarget     116625
HomeFouls      116584
AwayFouls      116584
HomeCorners    116194
AwayCorners    116194
HomeShots      115822
AwayShots      115819
dtype: int64

In [31]:
df = matches.copy()

# Keep only rows with result
df = df.dropna(subset=["FTResult"])

# Fill numeric NaNs with median
num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())


In [32]:
df["EloDiff"] = df["HomeElo"] - df["AwayElo"]
df["Form3Diff"] = df["Form3Home"] - df["Form3Away"]
df["Form5Diff"] = df["Form5Home"] - df["Form5Away"]

df["ShotDiff"] = df["HomeShots"] - df["AwayShots"]
df["TargetDiff"] = df["HomeTarget"] - df["AwayTarget"]

df["FoulDiff"] = df["HomeFouls"] - df["AwayFouls"]
df["CornerDiff"] = df["HomeCorners"] - df["AwayCorners"]

df["CardDiff"] = (df["HomeYellow"] + df["HomeRed"]) - (df["AwayYellow"] + df["AwayRed"])

df["GoalDiff"] = df["FTHome"] - df["FTAway"]

df["AbsEloDiff"] = abs(df["EloDiff"])
df["AbsForm5Diff"] = abs(df["Form5Diff"])
df["AbsShotDiff"] = abs(df["ShotDiff"])

df["TotalShots"] = df["HomeShots"] + df["AwayShots"]
df["PaceIndex"] = df["TotalShots"] / (df["HomeFouls"] + df["AwayFouls"] + 1)

df["LowScoringBias"] = (df["OddHome"] + df["OddAway"]) / df["OddDraw"]



In [33]:
df["ResultLabel"] = df["FTResult"].map({"H":0, "D":1, "A":2})
df["ResultLabel"].value_counts(normalize=True)


ResultLabel
0    0.446199
2    0.288696
1    0.265105
Name: proportion, dtype: float64

In [34]:
features = [
    "EloDiff","AbsEloDiff",
    "Form3Diff","AbsForm5Diff",
    "ShotDiff","AbsShotDiff",
    "TargetDiff",
    "CornerDiff","CardDiff",
    "LowScoringBias",
    "OddHome","OddDraw","OddAway"
]


X = df[features]
y = df["ResultLabel"]


In [40]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV
import numpy as np
import pandas as pd

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Class weights
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print("Class Weights:", class_weights)

sample_weights = y_train.map(class_weights)

# Model
xgb = XGBClassifier(
    n_estimators=900,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# Cross-validation (without calibration first)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

print("CV F1:", scores)
print("Mean CV:", scores.mean())

# Fit base model with weights
xgb.fit(X_train, y_train, sample_weight=sample_weights)

# Calibration
calibrated = CalibratedClassifierCV(
    estimator=xgb,
    method="isotonic",
    cv=3
)

calibrated.fit(X_train, y_train)

# Test
pred = calibrated.predict(X_test)
probs = calibrated.predict_proba(X_test)

print(classification_report(y_test, pred))


Class Weights: {np.int64(0): np.float64(0.7470533913339328), np.int64(1): np.float64(1.2573573020839723), np.int64(2): np.float64(1.1546161358173077)}
CV F1: [0.43763604 0.43464464 0.43764526 0.4402881  0.43722213]
Mean CV: 0.43748723513522625
              precision    recall  f1-score   support

           0       0.55      0.85      0.67     20575
           1       0.36      0.03      0.05     12224
           2       0.52      0.54      0.53     13312

    accuracy                           0.54     46111
   macro avg       0.48      0.47      0.42     46111
weighted avg       0.49      0.54      0.47     46111



In [43]:
#Feature importance
imp = pd.Series(xgb.feature_importances_, index=features)\
        .sort_values(ascending=False)

imp



TargetDiff        0.277644
OddAway           0.137062
OddHome           0.128135
CornerDiff        0.081933
ShotDiff          0.057076
OddDraw           0.051187
AbsShotDiff       0.043056
CardDiff          0.039818
LowScoringBias    0.039114
EloDiff           0.038648
AbsEloDiff        0.038333
Form3Diff         0.034430
AbsForm5Diff      0.033564
dtype: float32

In [44]:
#Predictions
df["PredictedResult"] = calibrated.predict(X)
df["PredictedLabel"] = df["PredictedResult"].map({0:"Home",1:"Draw",2:"Away"})

In [45]:
#Analytics Columns
df["TotalGoals"] = df["FTHome"] + df["FTAway"]
df["Over25Label"] = (df["TotalGoals"] > 2.5).astype(int)

df["ShotAccuracyHome"] = df["HomeTarget"] / df["HomeShots"]
df["ShotAccuracyAway"] = df["AwayTarget"] / df["AwayShots"]

In [46]:
#Export for tableau
df.to_csv("Data/Football_Tableau.csv", index=False)